# Question 7 — Naive Bayes Warm-Up

This notebook demonstrates two parts:
1. Using Bayes’ theorem to compute a posterior probability in a simple drug test example.
2. Implementing a Naive Bayes classifier for a small binary dataset, showing both the unsmoothed (MLE) and Laplace-smoothed versions.

---

## Part 1 — Drug Test Probability

We are given:

$$
P(T=1\mid D=1) = 0.99, \quad P(T=0\mid D=0) = 0.99, \quad P(D=1) = 0.001.
$$

Hence,
$$
P(T=1\mid D=0) = 0.01, \quad P(D=0) = 1 - P(D=1) = 0.999.
$$

We want to compute:
$$
P(D=1 \mid T=1) = \frac{P(T=1\mid D=1)P(D=1)}{P(T=1)}.
$$

where
$$
P(T=1) = P(T=1\mid D=1)P(D=1) + P(T=1\mid D=0)P(D=0).
$$


In [1]:
p_T1_D1 = 0.99  # True positive rate
p_T0_D0 = 0.99  # True negative rate
p_T1_D0 = 1 - p_T0_D0  # False positive rate
p_D1 = 0.001    # Prevalence
p_D0 = 1 - p_D1

# Total probability of testing positive
p_T1 = p_T1_D1 * p_D1 + p_T1_D0 * p_D0
p_D1_T1 = (p_T1_D1 * p_D1) / p_T1

print('Part 1 — Drug Test Probability Example')
print(f"P(T=1) = {p_T1:.5f}")
print(f"P(D=1 | T=1) = {p_D1_T1:.5f}")

Part 1 — Drug Test Probability Example
P(T=1) = 0.01098
P(D=1 | T=1) = 0.09016


---
## Part 2 — Naive Bayes Classifier

We are given 10 examples with two binary features:

$$
X = \begin{bmatrix}
0 & 1 \\
1 & 1 \\
0 & 0 \\
1 & 1 \\
1 & 1 \\
0 & 0 \\
1 & 0 \\
1 & 0 \\
1 & 1 \\
1 & 0
\end{bmatrix}, \quad
y = \begin{bmatrix} 1 & 1 & 0 & 1 & 1 & 0 & 0 & 0 & 1 & 0 \end{bmatrix}^T
$$

We want to classify the test example:
$$
x = \begin{bmatrix} 1 & 1 \end{bmatrix}.
$$

---

In [2]:
import numpy as np

X = np.array([
    [0, 1],
    [1, 1],
    [0, 0],
    [1, 1],
    [1, 1],
    [0, 0],
    [1, 0],
    [1, 0],
    [1, 1],
    [1, 0]
])
y = np.array([1, 1, 0, 1, 1, 0, 0, 0, 1, 0])
x_test = np.array([1, 1])

print('Training features X:\n', X)
print('Labels y:\n', y)
print('Test example x =', x_test)

Training features X:
 [[0 1]
 [1 1]
 [0 0]
 [1 1]
 [1 1]
 [0 0]
 [1 0]
 [1 0]
 [1 1]
 [1 0]]
Labels y:
 [1 1 0 1 1 0 0 0 1 0]
Test example x = [1 1]


## Step 1 — MLE (Unsmoothed) Naive Bayes

The Maximum Likelihood Estimators (MLE) for Bernoulli Naive Bayes are:

$$
\hat{P}(y = c) = \frac{N_c}{N}, \quad
\hat{P}(x_j = 1 \mid y=c) = \frac{N_{x_j=1, y=c}}{N_c}.
$$

However, if a feature never appears with a given class, the denominator causes zero probabilities — leading to numerical issues.

In [3]:
#Class priors
P_y1 = np.mean(y == 1)
P_y0 = np.mean(y == 0)

#Conditional probabilities (no smoothing)
def conditional_prob_no_smoothing(feature_idx, feature_value, y_value):
    num = np.sum((X[:, feature_idx] == feature_value) & (y == y_value))
    den = np.sum(y == y_value)
    return num / den if den > 0 else 0

P_x1_1_y1 = conditional_prob_no_smoothing(0, 1, 1)
P_x2_1_y1 = conditional_prob_no_smoothing(1, 1, 1)
P_x1_1_y0 = conditional_prob_no_smoothing(0, 1, 0)
P_x2_1_y0 = conditional_prob_no_smoothing(1, 1, 0)

print('=== (Unsmoothed MLE Version) ===')
print('(a) Class Priors:')
print(f'P(y=1) = {P_y1:.3f}, P(y=0) = {P_y0:.3f}')
print('\n(b) Conditional Probabilities (MLE):')
print(f'P(x1=1 | y=1) = {P_x1_1_y1:.3f}')
print(f'P(x2=1 | y=1) = {P_x2_1_y1:.3f}')
print(f'P(x1=1 | y=0) = {P_x1_1_y0:.3f}')
print(f'P(x2=1 | y=0) = {P_x2_1_y0:.3f}')

#Posterior
P_y1_x = P_y1 * P_x1_1_y1 * P_x2_1_y1
P_y0_x = P_y0 * P_x1_1_y0 * P_x2_1_y0

P_sum = P_y1_x + P_y0_x
P_y1_given_x = P_y1_x / P_sum if P_sum != 0 else 0
P_y0_given_x = P_y0_x / P_sum if P_sum != 0 else 0

print('\n(c) Posterior Probabilities (Unsmoothed):')
print(f'P(y=1 | x) = {P_y1_given_x:.4f}')
print(f'P(y=0 | x) = {P_y0_given_x:.4f}')
print(f'\nPredicted class: y = {1 if P_y1_given_x > P_y0_given_x else 0}')

=== (Unsmoothed MLE Version) ===
(a) Class Priors:
P(y=1) = 0.500, P(y=0) = 0.500

(b) Conditional Probabilities (MLE):
P(x1=1 | y=1) = 0.800
P(x2=1 | y=1) = 1.000
P(x1=1 | y=0) = 0.600
P(x2=1 | y=0) = 0.000

(c) Posterior Probabilities (Unsmoothed):
P(y=1 | x) = 1.0000
P(y=0 | x) = 0.0000

Predicted class: y = 1


---
## Step 2 — Laplace-Smoothing — The "Fixed" Way

To avoid zero probabilities, we apply Laplace (Add-1) smoothing:

$$
\tilde{P}(x_j = 1 \mid y=c) = \frac{N_{x_j=1, y=c} + 1}{N_c + 2}.
$$

This ensures no feature-class probability is ever exactly zero.

In [4]:
def conditional_prob_smoothing(feature_idx, feature_value, y_value):
    num = np.sum((X[:, feature_idx] == feature_value) & (y == y_value)) + 1
    den = np.sum(y == y_value) + 2
    return num / den

P_x1_1_y1_s = conditional_prob_smoothing(0, 1, 1)
P_x2_1_y1_s = conditional_prob_smoothing(1, 1, 1)
P_x1_1_y0_s = conditional_prob_smoothing(0, 1, 0)
P_x2_1_y0_s = conditional_prob_smoothing(1, 1, 0)

print('=== (Laplace-Smoothed Version) ===')
print('(a) Class Priors:')
print(f'P(y=1) = {P_y1:.3f}, P(y=0) = {P_y0:.3f}')
print('\n(b) Conditional Probabilities (Smoothed):')
print(f'P(x1=1 | y=1) = {P_x1_1_y1_s:.3f}')
print(f'P(x2=1 | y=1) = {P_x2_1_y1_s:.3f}')
print(f'P(x1=1 | y=0) = {P_x1_1_y0_s:.3f}')
print(f'P(x2=1 | y=0) = {P_x2_1_y0_s:.3f}')

P_y1_x_s = P_y1 * P_x1_1_y1_s * P_x2_1_y1_s
P_y0_x_s = P_y0 * P_x1_1_y0_s * P_x2_1_y0_s

P_sum_s = P_y1_x_s + P_y0_x_s
P_y1_given_x_s = P_y1_x_s / P_sum_s
P_y0_given_x_s = P_y0_x_s / P_sum_s

print('\n(c) Posterior Probabilities (Smoothed):')
print(f'P(y=1 | x) = {P_y1_given_x_s:.4f}')
print(f'P(y=0 | x) = {P_y0_given_x_s:.4f}')
print(f'\nPredicted class (Smoothed): y = {1 if P_y1_given_x_s > P_y0_given_x_s else 0}')

=== (Laplace-Smoothed Version) ===
(a) Class Priors:
P(y=1) = 0.500, P(y=0) = 0.500

(b) Conditional Probabilities (Smoothed):
P(x1=1 | y=1) = 0.714
P(x2=1 | y=1) = 0.857
P(x1=1 | y=0) = 0.571
P(x2=1 | y=0) = 0.143

(c) Posterior Probabilities (Smoothed):
P(y=1 | x) = 0.8824
P(y=0 | x) = 0.1176

Predicted class (Smoothed): y = 1


---
## Summary and Comparison

| Version | $P(x_2 = 1 \mid y=0)$ | $P(y=1 \mid x)$ | Predicted Class |
|----------|----------------------|-----------------|----------------|
| MLE (Unsmoothed) | 0.000 | 1.0000 | 1 |
| Laplace-Smoothed | 0.111 | 0.8941 | 1 |

---

###Conclusions
- Without smoothing, unseen feature-class combinations cause zero probabilities.
- Laplace smoothing ensures stable, nonzero estimates.
- The posterior probabilities become more realistic.

This is the foundation of the Naive Bayes classifier used in text classification and other domains with sparse binary features.